# 사고 연쇄 프롬프팅 (Chain-of-Thought Prompting)

사고 연쇄 프롬프팅은 모델이 복잡한 추론 과정을 단계적으로 진행하도록 유도하는 기법입니다. 이 방식은 특히 수학 문제, 논리 퍼즐, 복잡한 의사결정 등 여러 단계의 사고가 필요한 작업에 효과적입니다.

이 노트북에서는 사고 연쇄 프롬프팅의 기본 개념을 배우고 다양한 작업에 적용해보겠습니다.

In [1]:
# 필요한 라이브러리 및 모듈 임포트
import os
import sys
from dotenv import load_dotenv

# utils 디렉토리의 helpers.py 모듈을 임포트하기 위한 경로 설정
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils.helpers import get_completion, get_chat_completion

# .env 파일 로드
load_dotenv()

True

## 1. 사고 연쇄 프롬프팅의 기본 개념

사고 연쇄 프롬프팅은 모델에게 "단계적으로 생각해보세요" 또는 "이 문제를 단계별로 풀어보세요"와 같은 지시를 포함하여, 최종 답변을 내기 전에 추론 과정을 명시적으로 드러내도록 유도하는 기법입니다.

이 기법은 다음과 같은 장점이 있습니다:
1. 복잡한 문제에 대한 정확도 향상
2. 모델의 추론 과정을 확인할 수 있어 투명성 증가
3. 모델이 논리적 오류를 스스로 발견하고 수정할 기회 제공
4. 사용자가 모델의 사고 과정을 이해하고 학습할 수 있음

사고 연쇄 프롬프팅은 제로샷(Zero-shot) 방식과 퓨샷(Few-shot) 방식 모두 적용 가능합니다.

## 2. 제로샷 사고 연쇄 (Zero-shot Chain-of-Thought)

제로샷 사고 연쇄는 예시 없이 모델에게 직접 단계별로 생각하도록 지시하는 방식입니다. 주로 "단계별로 생각해보세요" 또는 "천천히 생각해보세요"와 같은 지시어를 사용합니다.

In [2]:
# 제로샷 사고 연쇄 프롬프팅 예시: 수학 문제
math_problem = """
철수는 사과 5개를 가지고 있었습니다. 영희가 철수에게 사과 2개를 주었고, 
그 후 철수는 민수에게 사과 3개를 주었습니다. 이제 철수에게 남은 사과는 몇 개인가요?
"""

# 일반적인 제로샷 프롬프팅
prompt_without_cot = f"다음 문제의 답을 구하세요: {math_problem}"

# 제로샷 사고 연쇄 프롬프팅
prompt_with_cot = f"""다음 문제의 답을 구하세요. 단계별로 천천히 생각해보세요: 
{math_problem}"""

# 두 방식 비교
response_without_cot = get_completion(prompt_without_cot)
print("=== 일반적인 제로샷 프롬프팅 결과 ===")
print(response_without_cot)
print("\n")

response_with_cot = get_completion(prompt_with_cot)
print("=== 제로샷 사고 연쇄 프롬프팅 결과 ===")
print(response_with_cot)

=== 일반적인 제로샷 프롬프팅 결과 ===
철수는 처음에 사과 5개를 가지고 있었습니다. 영희가 사과 2개를 주었으므로, 철수는 총 7개의 사과를 가지게 되었습니다. 그 후, 철수가 민수에게 사과 3개를 주었으니, 남은 사과는 7 - 3 = 4개입니다. 따라서, 철수에게 남은 사과는 4개입니다.


=== 제로샷 사고 연쇄 프롬프팅 결과 ===
문제를 단계별로 해결해 봅시다.

1. 처음에 철수는 사과 5개를 가지고 있었습니다.

2. 영희가 철수에게 사과 2개를 주었습니다.  
   - 그러면 철수의 사과는 \(5 + 2 = 7\)개가 됩니다.

3. 그 후 철수는 민수에게 사과 3개를 주었습니다.  
   - 그러면 철수의 사과는 \(7 - 3 = 4\)개가 됩니다.

따라서 이제 철수에게 남은 사과는 4개입니다.


## 3. 퓨샷 사고 연쇄 (Few-shot Chain-of-Thought)

퓨샷 사고 연쇄는 모델에게 추론 과정이 포함된 예시를 몇 가지 제공하여, 유사한 방식으로 새로운 문제를 해결하도록 유도하는 방식입니다.

In [3]:
# 퓨샷 사고 연쇄 프롬프팅 예시: 복잡한 수학 문제
prompt_few_shot_cot = """
다음은 복잡한 수학 문제를 단계별로 해결하는 예시입니다:

문제: 철수는 빵을 8개 가지고 있었습니다. 영희에게 2개를 주고, 민수에게 빵의 절반을 주었습니다. 이제 철수에게 남은 빵은 몇 개인가요?
풀이: 
1. 처음에 철수는 빵 8개를 가지고 있었습니다.
2. 영희에게 2개를 주었으므로, 8 - 2 = 6개가 남았습니다.
3. 민수에게 남은 빵의 절반을 주었으므로, 6 ÷ 2 = 3개를 주었습니다.
4. 따라서 철수에게 남은 빵은 6 - 3 = 3개입니다.
답: 3개

문제: 영희는 40페이지짜리 책을 읽고 있습니다. 첫날에 전체 페이지의 1/4을 읽었고, 둘째 날에는 남은 페이지의 1/3을 읽었습니다. 영희가 아직 읽지 않은 페이지는 몇 페이지인가요?
풀이:
1. 책은 총 40페이지입니다.
2. 첫날에 전체 페이지의 1/4을 읽었으므로, 40 × 1/4 = 10페이지를 읽었습니다.
3. 첫날 이후 남은 페이지는 40 - 10 = 30페이지입니다.
4. 둘째 날에는 남은 페이지의 1/3을 읽었으므로, 30 × 1/3 = 10페이지를 더 읽었습니다.
5. 따라서 총 읽은 페이지는 10 + 10 = 20페이지이고, 읽지 않은 페이지는 40 - 20 = 20페이지입니다.
답: 20페이지

이제 다음 문제를 위와 같이 단계별로 풀어보세요:

문제: 학교 축제에서 300개의 풍선이 있었습니다. 오전에 풍선의 2/5이 사용되었고, 오후에는 남은 풍선의 1/3이 사용되었습니다. 그 후 학생들이 실수로 풍선 10개를 터뜨렸습니다. 남은 풍선은 몇 개인가요?
풀이:
"""

response = get_completion(prompt_few_shot_cot)
print(response)

풀이:
1. 처음에 학교 축제에는 300개의 풍선이 있었습니다.
2. 오전에 풍선의 2/5이 사용되었으므로, 300 × 2/5 = 120개의 풍선이 사용되었습니다.
3. 오전 이후 남은 풍선은 300 - 120 = 180개입니다.
4. 오후에는 남은 풍선의 1/3이 사용되었으므로, 180 × 1/3 = 60개의 풍선이 사용되었습니다.
5. 오후 이후 남은 풍선은 180 - 60 = 120개입니다.
6. 학생들이 실수로 풍선 10개를 터뜨렸으므로, 120 - 10 = 110개가 남았습니다.
답: 110개


## 4. 복잡한 추론 문제 해결하기

사고 연쇄 프롬프팅은 단순한 수학 문제뿐만 아니라 복잡한 추론이 필요한 다양한 문제에 적용할 수 있습니다. 다음은 논리 퍼즐에 적용한 예시입니다.

In [4]:
# 복잡한 논리 퍼즐에 사고 연쇄 프롬프팅 적용
logic_puzzle = """
다섯 명의 친구 A, B, C, D, E가 한 줄로 서 있습니다. 다음과 같은 조건이 있을 때, 왼쪽에서 오른쪽으로 어떤 순서로 서 있는지 구하세요.

1. A는 B의 왼쪽에 있다.
2. C는 맨 오른쪽에 있지 않다.
3. D는 맨 왼쪽에 있지 않다.
4. E는 D의 오른쪽에 있다.
5. B와 E 사이에는 정확히 한 명이 있다.
"""

prompt = f"""
다음 논리 퍼즐을 단계별로 해결해보세요. 모든 가능성을 체계적으로 검토하고, 
각 단계에서 어떤 조건이 만족되는지 또는 만족되지 않는지 명확히 설명해주세요.

{logic_puzzle}
"""

response = get_completion(prompt)
print(response)

이 논리 퍼즐을 해결하기 위해 주어진 조건들을 체계적으로 검토해보겠습니다.

1. **조건 검토**:
   - A는 B의 왼쪽에 있다.
   - C는 맨 오른쪽에 있지 않다.
   - D는 맨 왼쪽에 있지 않다.
   - E는 D의 오른쪽에 있다.
   - B와 E 사이에는 정확히 한 명이 있다.

2. **가능성 탐색**:
   - 총 다섯 자리: \(\_\_\_\_\_\)

3. **조건 3 적용**: D는 맨 왼쪽에 있을 수 없다. 따라서 D는 두 번째, 세 번째, 네 번째, 또는 다섯 번째 자리에 있을 수 있습니다.

4. **조건 2 적용**: C는 맨 오른쪽에 있을 수 없다. 따라서 C는 첫 번째, 두 번째, 세 번째, 또는 네 번째 자리에 있을 수 있습니다.

5. **조건 5 적용**: B와 E 사이에 정확히 한 명이 있어야 하므로 가능한 배열은 (B, \_, E) 또는 (E, \_, B)입니다.

6. **조건 4 적용**: E는 항상 D의 오른쪽에 있어야 합니다. 따라서 (B, \_, E) 배열에서 B가 D의 오른쪽에 있으면 안 됩니다. 

7. **조건 1 적용**: A는 B의 왼쪽에 있어야 하므로 B의 오른쪽에는 A가 있을 수 없습니다.

8. **가능한 배열 조합**:

   - **조합 1**: (A, B, \_, E, D)
   - **조합 2**: (A, B, D, E, \_)
   - **조합 3**: (A, \_, B, \_, E)

9. **조합 1 검토**: (A, B, \_, E, D)
   - E는 D의 오른쪽에 있으므로 맞지 않습니다.

10. **조합 2 검토**: (A, B, D, E, \_)
    - B와 E 사이에 D가 있으나 E가 D의 오른쪽에 있으므로 조건에 부합합니다.
    - C는 맨 오른쪽에 있지 않으므로 C는 네 번째 자리에 와야 합니다. 이 조합은 유효합니다.

11. **조합 3 검토**: (A, \_, B, \_, E)
    - B와 E 사이에 정확히 한 명이 있어야 하므로, D가 들어가야 

## 5. 사고 연쇄 프롬프팅의 응용: 알고리즘 설계

사고 연쇄 프롬프팅은 알고리즘 설계와 같은 복잡한 프로그래밍 작업에도 적용할 수 있습니다.

In [5]:
# 알고리즘 설계에 사고 연쇄 프롬프팅 적용
algorithm_problem = """
최장 증가 부분 수열(Longest Increasing Subsequence)의 길이를 구하는 알고리즘을 설계하세요.
예를 들어, 배열 [10, 9, 2, 5, 3, 7, 101, 18]의 가장 긴 증가하는 부분 수열은 [2, 3, 7, 18] 또는 [2, 5, 7, 101]이므로 길이는 4입니다.
"""

prompt = f"""
다음 알고리즘 문제를 해결해보세요. 다음 단계를 따라 접근해주세요:
1. 문제를 이해하고 분석하세요.
2. 간단한 예시로 문제를 테스트해보세요.
3. 가능한 접근 방법을 여러 가지 생각해보세요 (브루트 포스, 그리디, 동적 프로그래밍 등).
4. 가장 효율적인 접근 방법을 선택하고 알고리즘을 설계하세요.
5. 알고리즘의 의사코드나 실제 코드를 작성하세요.
6. 시간 복잡도와 공간 복잡도를 분석하세요.
7. 알고리즘을 테스트하고 검증하세요.

{algorithm_problem}
"""

response = get_completion(prompt)
print(response)

1. **문제 이해 및 분석**:
   - 주어진 배열에서 가장 긴 증가 부분 수열(Longest Increasing Subsequence, LIS)의 길이를 구하는 문제입니다.
   - 증가 부분 수열이란 각 원소가 이전 원소보다 큰 형태로 배열된 부분 배열입니다.
   - 예를 들어, 배열 [10, 9, 2, 5, 3, 7, 101, 18]에서 LIS는 [2, 3, 7, 18] 또는 [2, 5, 7, 101]이고, 길이는 4입니다.

2. **간단한 예시로 테스트**:
   - 배열 [3, 10, 2, 1, 20]의 경우 LIS는 [3, 10, 20]이고 길이는 3입니다.
   - 배열 [3, 2]의 경우 LIS는 [3] 또는 [2]이고 길이는 1입니다.
   - 배열 [50, 3, 10, 7, 40, 80]의 경우 LIS는 [3, 7, 40, 80]이고 길이는 4입니다.

3. **가능한 접근 방법**:
   - **브루트 포스**: 모든 부분 수열을 생성하고 그중에서 증가하는 수열을 찾아 가장 긴 것을 선택하는 방법. 매우 비효율적입니다.
   - **동적 프로그래밍(DP)**: 각 위치까지의 부분 문제를 해결하면서 LIS 길이를 점진적으로 계산하는 방법.
   - **이진 탐색을 이용한 방법**: DP와 이진 탐색을 결합하여 보다 효율적으로 LIS를 찾는 방법.

4. **효율적인 접근 방법 선택 및 알고리즘 설계**:
   - 동적 프로그래밍과 이진 탐색을 결합한 방법이 시간 복잡도 O(n log n)으로 가장 효율적입니다.

5. **의사코드 작성**:
   ```plaintext
   function lengthOfLIS(nums):
       if nums is empty:
           return 0
       
       dp = an empty list
       
       for num in nums:
           if dp is empty or num > last element in dp:
           

## 6. 채팅 형식의 사고 연쇄 프롬프팅

채팅 모델을 사용할 때는 대화 형식으로 사고 연쇄 프롬프팅을 적용할 수도 있습니다.

In [6]:
# 채팅 형식의 사고 연쇄 프롬프팅
messages = [
    {"role": "system", "content": "당신은 단계별로 생각하며 복잡한 문제를 해결하는 전문가입니다. 각 단계를 명확히 설명하고 논리적으로 문제에 접근해주세요."},
    {"role": "user", "content": "다음 윤리적 딜레마에 대해 생각해보세요: '만약 누군가의 생명을 구하기 위해 거짓말을 해야 한다면, 당신은 거짓말을 할 것인가요? 왜 그렇게 생각하나요?'"}
]

response = get_chat_completion(messages)
print(response)

이 윤리적 딜레마는 진실성과 생명 보호라는 두 가지 중요한 가치 사이의 갈등을 포함하고 있습니다. 이러한 상황을 분석하기 위해 몇 가지 윤리적 이론과 관점을 고려할 수 있습니다.

1. **결과주의적 접근 (공리주의)**:
   - 공리주의는 결과에 따라 행위의 도덕성을 평가합니다. 이 관점에서 보면, 거짓말이 한 사람의 생명을 구할 수 있다면 이는 긍정적인 결과를 가져오기 때문에 도덕적으로 허용될 수 있습니다. 생명을 구하는 것이 더 큰 행복과 이익을 가져오기 때문입니다.

2. **의무론적 접근 (칸트주의)**:
   - 칸트주의 윤리학에서는 행위의 도덕성을 결과가 아닌 행위 자체의 원칙에 따라 평가합니다. 칸트는 거짓말을 어떤 상황에서도 허용할 수 없는 것으로 보았습니다. 이는 거짓말이 보편화될 경우 사회적 신뢰를 무너뜨릴 수 있기 때문입니다. 따라서, 생명을 구하기 위해서라도 거짓말을 하는 것은 도덕적으로 옳지 않다는 결론에 도달할 수 있습니다.

3. **덕 윤리적 접근**:
   - 덕 윤리에서는 행위보다는 행위자의 성품과 덕목에 초점을 맞춥니다. 여기서 중요한 덕목은 자비와 정직입니다. 생명을 구하기 위해 거짓말을 하는 것은 자비로운 행동으로 볼 수 있지만, 정직의 덕목을 손상시킬 수도 있습니다. 따라서, 상황에 따라 어느 덕목이 더 중요한지를 판단해야 합니다.

4. **상황 윤리**:
   - 상황 윤리는 구체적인 상황에 따라 다른 윤리적 결정을 내릴 수 있음을 강조합니다. 이 관점에서는 생명을 구하는 것이 최우선이라면 특정 상황에서는 거짓말이 정당화될 수 있습니다. 그러나 이 경우에도 가능한 한 덜 해로운 거짓말을 선택해야 합니다.

**개인적 판단**:
- 개인적으로는 생명을 구하는 것이 매우 중요한 가치이기 때문에, 특정 상황에서는 거짓말을 선택할 수 있습니다. 그러나 이는 신중한 판단이 필요하며, 거짓말로 인한 다른 사람의 피해와 신뢰 손실을 최소화하기 위한 노력이 필요합니다. 또한, 이후의 신뢰 회복을 위한 노력도 중요합니다.

이와 

## 7. 실습: 다양한 문제에 사고 연쇄 프롬프팅 적용하기

이제 여러분이 직접 다양한 문제에 사고 연쇄 프롬프팅을 적용해보세요. 다음은 실습해볼 수 있는 문제들입니다:

1. 복잡한 수학 문제
2. 논리 퍼즐이나 추리 게임
3. 윤리적 딜레마나 의사결정 문제
4. 프로그래밍 알고리즘 설계
5. 텍스트 분석이나 해석이 필요한 문제

각 문제에 대해 효과적인 사고 연쇄 프롬프트를 만들어보고, 결과를 분석해보세요.

In [7]:
# 여기에 여러분의 사고 연쇄 프롬프팅 실험을 작성하세요
# 예: 복잡한 의사결정 문제
problem = """
당신은 소규모 스타트업의 CEO입니다. 회사가 성장하면서 다음 세 가지 옵션 중 하나를 선택해야 합니다:
1. 대기업으로부터의 인수 제안 수락 (즉시 큰 금액 보장, 하지만 비전과 자율성 상실 위험)
2. 벤처 캐피털로부터의 대규모 투자 유치 (성장 가능성, 하지만 투자자 압박과 기대 증가)
3. 현재 상태 유지하며 유기적 성장 (자율성 유지, 하지만 성장 속도 둔화와 경쟁 위험)

각 옵션의 장단점을 고려하여, 어떤 선택이 최선인지 결정하세요.
"""

prompt = f"""
다음 의사결정 문제를 단계별로 분석해주세요. 다음 과정을 따라주세요:
1. 각 옵션의 장점을 자세히 분석하세요.
2. 각 옵션의 단점과 위험 요소를 자세히 분석하세요.
3. 단기적 관점과 장기적 관점에서 각 옵션을 평가하세요.
4. 주요 이해관계자(창업자, 직원, 고객, 투자자 등)에게 미치는 영향을 고려하세요.
5. 최종 결정을 내리고 그 이유를 설명하세요.

{problem}
"""

# 프롬프트 실행 및 결과 분석
response = get_completion(prompt)
print(response)

이 의사결정 문제를 단계별로 분석해보겠습니다.

### 1. 각 옵션의 장점 분석

**옵션 1: 대기업으로부터의 인수 제안 수락**
- **즉각적인 재정적 보상**: 인수 제안을 수락하면 즉시 큰 금액이 보장됩니다. 이는 창업자와 초기 투자자들에게 재정적 안정성을 제공합니다.
- **자원 및 인프라**: 대기업의 자원을 활용할 수 있어 운영 효율성이 증가할 수 있습니다.
- **시장 접근성**: 대기업의 네트워크를 활용하여 더 광범위한 시장에 접근할 수 있습니다.

**옵션 2: 벤처 캐피털로부터의 대규모 투자 유치**
- **성장 자본 확보**: 대규모 자본을 통해 제품 개발, 마케팅, 인력 확장 등 다양한 성장을 도모할 수 있습니다.
- **경쟁력 강화**: 추가 자본을 통해 경쟁사보다 유리한 위치를 점할 수 있습니다.
- **독립성 유지**: 인수 제안과 달리, 회사의 독립성을 유지하며 성장할 수 있습니다.

**옵션 3: 현재 상태 유지하며 유기적 성장**
- **자율성 유지**: 외부의 간섭 없이 창업자의 비전과 목표에 따라 회사를 운영할 수 있습니다.
- **리스크 관리 용이**: 외부 자본에 의한 압박 없이 리스크를 관리할 수 있습니다.
- **문화 보존**: 회사의 문화를 유지하며 직원들의 사기를 높일 수 있습니다.

### 2. 각 옵션의 단점과 위험 요소 분석

**옵션 1: 대기업으로부터의 인수 제안 수락**
- **비전 상실**: 대기업의 전략에 따라 회사의 비전이 변질될 수 있습니다.
- **자율성 상실**: 경영 및 운영상의 자율성을 잃고 대기업의 지시에 따라야 할 수 있습니다.
- **직원 이탈 위험**: 회사 문화가 변화하면서 직원들이 이탈할 가능성이 있습니다.

**옵션 2: 벤처 캐피털로부터의 대규모 투자 유치**
- **투자자 압박**: 투자자들의 기대에 부응해야 하며, 이는 단기적 성과 압박으로 이어질 수 있습니다.
- **지분 희석**: 추가 투자를 받으면서 창업자 및 초기 투자자의 지분이 희석될 수 있습니다.
- **의

## 8. 사고 연쇄 프롬프팅의 최적화 팁

효과적인 사고 연쇄 프롬프팅을 위한 몇 가지 팁을 정리해보았습니다:

1. **명확한 단계 설정**: "다음 단계를 따라 생각해보세요"와 같이 구체적인 단계를 제시합니다.
2. **충분한 사고 시간 부여**: "천천히 생각해보세요"와 같은 표현으로 모델이 충분히 추론하도록 유도합니다.
3. **중간 결과 확인**: 복잡한 문제는 중간 결과를 확인하고 검증하도록 지시합니다.
4. **다양한 접근법 고려**: 한 가지 방법으로 풀지 못하면 다른 방법을 시도하도록 유도합니다.
5. **예시 제공**: 유사한 문제의 해결 과정 예시를 제공하면 더 효과적입니다.
6. **오류 자가 검증**: 최종 답변 전에 논리적 오류가 없는지 확인하도록 지시합니다.

이러한 팁을 활용하여 사고 연쇄 프롬프팅의 효과를 극대화할 수 있습니다.

## 9. 요약 및 다음 단계

이 노트북에서는 사고 연쇄 프롬프팅의 기본 개념과 다양한 적용 방법을 살펴보았습니다. 사고 연쇄 프롬프팅은 모델이 복잡한 문제를 단계적으로 해결하도록 유도하는 강력한 기법입니다.

다음 노트북에서는 자기 일관성(Self-Consistency) 기법에 대해 알아볼 예정입니다. 자기 일관성은 사고 연쇄 프롬프팅을 확장하여 여러 추론 경로를 생성한 후 가장 일관된 답변을 선택하는 방식입니다.

실습을 통해 배운 내용:
- 제로샷 사고 연쇄 프롬프팅 방법
- 퓨샷 사고 연쇄 프롬프팅 방법
- 복잡한 추론 문제에 적용하기
- 알고리즘 설계에 적용하기
- 채팅 형식의 사고 연쇄 프롬프팅

이러한 기법을 실제 작업에 적용하여 LLM의 성능을 최적화해보세요!